# RAG di immagini — cercare e interrogare un archivio di immagini con il linguaggio naturale

*Notebook didattico per VS Code / Jupyter — © 2026 Antonio Piemontese*

---

## Di che cosa parliamo

**RAG** è l'acronimo di *Retrieval-Augmented Generation*, letteralmente «generazione aumentata dal recupero». È l'architettura con cui facciamo rispondere un modello linguistico **basandosi su un archivio nostro** invece che solo sulla sua memoria interna. Il ciclo è sempre lo stesso, in tre passi:

1. **Retrieval** (recupero): data una domanda, cerchiamo nell'archivio i contenuti più pertinenti;
2. **Augmentation** (arricchimento): "arricchiamo" la domanda allegandole i contenuti recuperati;
3. **Generation** (generazione): un modello produce la risposta leggendo **solo** ciò che gli abbiamo allegato.

Di solito applichiamo la RAG a documenti di **testo**. In questo notebook la estendiamo alle **immagini**: parliamo quindi di RAG **multimodale**, dove "multimodale" indica un sistema che tratta più *modalità* di dato (testo, immagini, audio, video...) nello stesso spazio di lavoro.

### Perché ci interessa, in banca e in assicurazione

Gli archivi aziendali non contengono solo testo: pensiamo alle **perizie fotografiche dei sinistri** ("trovami le foto dei danni da grandine del 2025"), agli **schemi di processo** nei manuali interni, ai **grafici** sparsi in centinaia di report. Poter cercare queste immagini *scrivendo una frase in italiano* — e poi farle commentare a un modello — è esattamente ciò che costruiamo qui.

### La pipeline che costruiremo

Chiamiamo **pipeline** la catena di passaggi che trasforma l'input nell'output. La nostra è questa:

```text
                 (indicizzazione: la facciamo UNA volta)
  immagini ──► embedding immagine ──► database vettoriale
                                            ▲
                                            │ ricerca dei più simili
  domanda in italiano ──► embedding testo ──┘──► immagini recuperate ──► modello multimodale ──► risposta
```

I termini nuovi (*embedding*, *database vettoriale*, *modello multimodale*) li definiremo uno per uno, la prima volta che li incontriamo.

### Indice

1. **Il dataset** — generiamo con Pillow e Matplotlib dodici immagini "ad hoc" (forme, scene, grafici)
2. **Gli embedding multimodali** — trasformiamo immagini e testi in vettori confrontabili, con CLIP
3. **La ricerca "as-is"** — similarità coseno calcolata a mano con NumPy, per capire i meccanismi
4. **La versione idiomatica** — un database vettoriale (ChromaDB), con metadati e filtri
5. **Ricerca per immagine** — la query non è una frase, ma un'altra immagine
6. **La "G" di RAG** — la risposta finale con un modello multimodale (gpt-4o-mini)
7. **Per andare oltre** — esercizi e varianti

In chiusura, una sintesi e un piccolo glossario.

---

**Legenda delle icone (standard) usate nel notebook**:<br>

👉 punto di attenzione, il "succo"<br>
📌 nota / inizio di una nota<br>
📦 punto elenco importante<br>
📊 dati/numeri<br>
🔹 punto elenco normale<br>
⭐ punto elenco importante<br>
✅ punto risolto, positivo<br>
❌ punto negativo, da evitare<br>
⚠️ attenzione, warning, allarme<br>
💡 idea chiave<br>
🧠 idea intuitiva<br>
📝 sintesi / bottom-line<br>
⟶ conseguenza, effetto, passo successivo

## Preparazione dell'ambiente

Lavoriamo in **VS Code** con le estensioni *Python* e *Jupyter* installate. Il notebook funziona anche in Jupyter "classico" e su Google Colab.

### Creazione dell'ambiente conda (Windows, PowerShell)

Creiamo un ambiente dedicato. **In questo ambiente deroghiamo alla nostra consueta strategia "prima conda, poi pip"**: PyTorch si installa comunque con pip (è la nostra eccezione standard) e, per evitare il conflitto OpenMP/MKL già incontrato in passato (il kernel Jupyter che si chiude sulla prima cella Matplotlib eseguita dopo torch), installiamo con pip **tutto** lo stack numerico, così da avere build coerenti (OpenBLAS). Come sempre, prima dei `pip install` impostiamo `PYTHONNOUSERSITE=1` per escludere lo *user site* (la cartella di pacchetti personale di Windows, fuori dagli ambienti, che in passato ci ha "contaminato" gli ambienti conda).

```cmd
    conda create -n rag-immagini python=3.12 -y
    conda activate rag-immagini
    set PYTHONNOUSERSITE=1
    python -m pip install numpy pandas matplotlib pillow sentence-transformers chromadb openai python-dotenv
```

oppure:

```powershell
    conda create -n rag-immagini python=3.12 -y
    conda activate rag-immagini
    $env:PYTHONNOUSERSITE = "1"
    python -m pip install numpy pandas matplotlib pillow sentence-transformers chromadb openai python-dotenv
```

👉 **Il comando *cmd* `set` ha alcune trappole classiche**:
- il comando *cmd* `set NOME=valore` è l’equivalente di $env:NOME = "valore" in PowerShell: <u>vale solo per quella finestra e per i processi lanciati da lì</u> — esattamente ciò che ci serve, purché venga eseguito **prima del `pip install` nella stessa finestra**;
- niente spazi attorno all’=: a differenza di PowerShell, `set PYTHONNOUSERSITE = 1` creerebbe una variabile chiamata `PYTHONNOUSERSITE` (con lo spazio finale nel nome!) e il nostro scudo anti-contaminazione fallirebbe in silenzio;
- niente virgolette: in *cmd* finirebbero dentro il valore (`set X="1"` memorizza proprio "1", virgolette comprese) ⟶  verificare la variabile, se non è definita stampa "variabile di ambiente non definita».

---
> 📌 **Nota GPU.** Su Windows, `pip install torch` (che `sentence-transformers` si porta dietro come dipendenza) installa la build **solo CPU**. Per questo notebook la CPU basta e avanza: **i modelli sono piccoli**. Se però vogliamo usare la GPU (es. RTX serie 50, architettura `sm_120`), installiamo **prima** PyTorch con la build CUDA e **poi** il resto, così pip non lo sostituisce:
>
> ```powershell
> python -m pip install torch --index-url https://download.pytorch.org/whl/cu130
> python -m pip install numpy pandas matplotlib pillow sentence-transformers chromadb openai python-dotenv
> ```

---
### Selezione del kernel in VS Code

Aperto il notebook, in alto a destra scegliamo **Select Kernel ▸ Python Environments ▸ rag-immagini**. Il **kernel** è il processo Python che esegue le celle: deve essere quello dell'ambiente in cui abbiamo appena installato i pacchetti.

> 🌐 **Connessione.** Alla prima esecuzione il notebook scarica due modelli da Hugging Face (circa 1 GB in totale, una volta sola: finiscono nella *cache*, una cartella di riuso in `~/.cache`). Dopo il primo giro funziona tutto offline, tranne la cella finale che chiama l'API di OpenAI (facoltativa).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Installazione dei pacchetti DENTRO il notebook (facoltativa)
# ─────────────────────────────────────────────────────────────────────────────
# Questa cella serve solo su Google Colab, o se non abbiamo preparato l'ambiente
# conda come descritto sopra. In locale, con l'ambiente "rag-immagini" già
# pronto, la saltiamo.
#
# Usiamo il comando "magico" %pip (i comandi magici sono istruzioni speciali di
# Jupyter che iniziano con %): a differenza di !pip, %pip installa i pacchetti
# nel kernel ATTIVO, cioè proprio nell'interprete che sta eseguendo le celle.
#
# Per usarla: togliamo il cancelletto alla riga seguente ed eseguiamo.

# %pip install -q numpy pandas matplotlib pillow sentence-transformers chromadb openai python-dotenv

👉 La convenzione “tutti gli import in testa” (PEP 8) vale per i moduli e gli script, dove il file è un’unità eseguita tutta insieme; **in un notebook didattico il file è un percorso a capitoli, e importare all’inizio del capitolo i package di cui ha bisogno è legittimo e anzi <u>più chiaro</u>**. 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Librerie di base e impostazioni generali
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path          # percorsi di file e cartelle in modo portabile
import math                       # funzioni matematiche (seno, coseno: ci serviranno per la stella)

import numpy as np                # calcolo numerico su array (vettori e matrici)
import pandas as pd               # tabelle di dati (DataFrame): le usiamo per il catalogo
import matplotlib.pyplot as plt   # grafici
from PIL import Image, ImageDraw  # Pillow: creazione e modifica di immagini

import PIL, matplotlib            # solo per stampare le versioni qui sotto

# Stampiamo le versioni: quando un notebook "non funziona più", la prima cosa
# da controllare è quasi sempre la versione delle librerie.
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("Pillow     :", PIL.__version__)

# Il "seme" (seed) inizializza il generatore di numeri casuali: fissandolo,
# ogni esecuzione produce GLI STESSI numeri "casuali". È la base della
# riproducibilità: noi e i partecipanti vedremo tutti le stesse immagini.
SEME = 42
rng = np.random.default_rng(SEME)   # generatore di numeri casuali di NumPy

## 1. Il dataset: generiamo le immagini «ad hoc»

In un progetto reale partiremmo da un archivio esistente (foto di perizie, schemi, grafici estratti dai report). In aula, però, vogliamo un notebook **autosufficiente e riproducibile**: per questo il dataset ce lo **generiamo da soli**, con due librerie:

- **Pillow** (il nome del pacchetto è l'erede di *PIL*, Python Imaging Library): lo standard di fatto per creare e modificare immagini in Python, disegnando forme su una "tela";
- **Matplotlib**: la libreria dei grafici, che qui usiamo per produrre... immagini di grafici.

Un dataset costruito artificialmente si chiama **dataset sintetico**: non proviene dal mondo reale, ma lo imita quanto basta per l'esercizio.

> 📌 **Il sistema di coordinate di Pillow.** Attenzione, è contro-intuitivo: l'origine `(0, 0)` è **in alto a sinistra**, la `x` cresce verso destra e la `y` cresce **verso il basso**. Quindi "alzare" un punto significa *diminuire* la sua `y`. È la convenzione tipica della grafica digitale (deriva dall'ordine con cui gli schermi disegnano le righe).

### Le tre categorie

| Categoria | Immagini | Perché ci serve |
|---|---|---|
| `forme`   | cerchio rosso, quadrato blu, triangolo verde, stella gialla | test elementari su **colore** e **forma** |
| `scene`   | casa, albero con sole, barca a vela, automobile | piccole "scene" riconoscibili, più vicine a una foto |
| `grafici` | barre, linee, torta, dispersione | il caso aziendale tipico: figure dentro i report |

### Il catalogo

Per ogni immagine registriamo tre informazioni: il **file**, la **categoria** e una **descrizione** in italiano. La descrizione è la nostra *ground truth* («verità di riferimento»): sappiamo noi, in anticipo, che cosa c'è in ogni immagine, e questo ci permetterà di giudicare se la ricerca funziona. Categoria e descrizione diventeranno i **metadati** dell'indice (ne riparleremo al §4).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Costanti del dataset
# ─────────────────────────────────────────────────────────────────────────────
IMG_DIR = Path("immagini")        # cartella di destinazione delle immagini
IMG_DIR.mkdir(exist_ok=True)      # exist_ok=True: se esiste già non è un errore
                                  # (così la cella si può rieseguire senza problemi)

LATO = 400                        # immagini quadrate di 400×400 pixel: più che
                                  # sufficienti, perché il modello che useremo
                                  # le ridimensiona comunque a 224×224

BIANCO = (255, 255, 255)          # i colori in Pillow sono triple RGB
                                  # (Rosso, Verde, Blu), ciascuno da 0 a 255

# Palette dei nostri materiali didattici (teal / coral / purple / gray),
# verificata per l'accessibilità: i quattro colori restano distinguibili anche
# con i principali deficit di visione dei colori (daltonismo). La usiamo per i
# GRAFICI; le forme geometriche usano invece colori "canonici" (rosso, blu...),
# perché è proprio sul nome del colore che vogliamo interrogare la ricerca.
PALETTE = {
    "teal":   "#0A9FBE",
    "coral":  "#E8604F",
    "purple": "#8457C8",
    "gray":   "#4E5660",
}

INCHIOSTRO = "#2B2B2B"            # colore del testo nei grafici (quasi nero):
                                  # il testo resta neutro, il colore identifica le serie

def nuova_tela(sfondo=BIANCO):
    # Creiamo una "tela" (un'immagine vuota) e il suo "pennello".
    # "RGB" è la modalità colore; ImageDraw.Draw(img) restituisce l'oggetto
    # con i metodi di disegno (rectangle, ellipse, polygon, line, ...).
    img = Image.new("RGB", (LATO, LATO), sfondo)
    return img, ImageDraw.Draw(img)

def salva(img, nome_file, categoria, descrizione):
    # Salviamo l'immagine su disco e restituiamo la sua "voce di catalogo":
    # un dizionario con le tre informazioni che ci servono (file, categoria,
    # descrizione = la nostra ground truth).
    percorso = IMG_DIR / nome_file
    img.save(percorso)
    return {"file": str(percorso), "categoria": categoria, "descrizione": descrizione}

print(f"Le immagini verranno salvate in: {IMG_DIR.resolve()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Categoria 1: le FORME geometriche
# ─────────────────────────────────────────────────────────────────────────────
# Ogni funzione disegna su una tela bianca e restituisce la voce di catalogo.
# Le coordinate di rettangoli ed ellissi si danno come "riquadro di ingombro":
# [x_sinistra, y_alto, x_destra, y_basso].

def genera_forme():
    voci = []

    # Cerchio rosso: ellipse() dentro un riquadro QUADRATO produce un cerchio.
    img, pennello = nuova_tela()
    pennello.ellipse([80, 80, 320, 320], fill=(214, 40, 40))
    voci.append(salva(img, "cerchio_rosso.png", "forme",
                      "Un cerchio rosso su sfondo bianco"))

    # Quadrato blu: un semplice rettangolo con lati uguali.
    img, pennello = nuova_tela()
    pennello.rectangle([90, 90, 310, 310], fill=(40, 80, 214))
    voci.append(salva(img, "quadrato_blu.png", "forme",
                      "Un quadrato blu su sfondo bianco"))

    # Triangolo verde: polygon() unisce una lista di vertici (x, y).
    # Ricordiamo: la y cresce verso il basso, quindi il vertice "in alto"
    # è quello con la y PIÙ PICCOLA (200, 70).
    img, pennello = nuova_tela()
    pennello.polygon([(200, 70), (60, 330), (340, 330)], fill=(36, 150, 66))
    voci.append(salva(img, "triangolo_verde.png", "forme",
                      "Un triangolo verde su sfondo bianco"))

    # Stella gialla: la costruiamo con un po' di trigonometria.
    # Alterniamo 5 punte su un cerchio ESTERNO e 5 rientranze su uno INTERNO,
    # girando di 36° alla volta (360° / 10 vertici). Partiamo dall'angolo di
    # 90° (π/2) così la prima punta è rivolta verso l'alto; usiamo "cy - sin"
    # perché, di nuovo, la y della tela cresce verso il basso.
    img, pennello = nuova_tela()
    cx, cy = 200, 210                      # centro della stella
    r_esterno, r_interno = 150, 60         # raggio delle punte e delle rientranze
    vertici = []
    for i in range(10):
        r = r_esterno if i % 2 == 0 else r_interno
        angolo = math.pi / 2 + i * math.pi / 5
        vertici.append((cx + r * math.cos(angolo), cy - r * math.sin(angolo)))
    pennello.polygon(vertici, fill=(245, 192, 20))
    voci.append(salva(img, "stella_gialla.png", "forme",
                      "Una stella gialla su sfondo bianco"))

    return voci

voci_forme = genera_forme()
print(f"Generate {len(voci_forme)} forme:",
      ", ".join(Path(v["file"]).name for v in voci_forme))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Categoria 2: le SCENE stilizzate
# ─────────────────────────────────────────────────────────────────────────────
# Qui componiamo più forme per ottenere piccole "scene" riconoscibili.
# L'ordine di disegno conta: ciò che disegniamo DOPO copre ciò che c'era prima
# (prima lo sfondo, poi gli oggetti, infine i dettagli).

CIELO = (173, 214, 240)   # azzurro
PRATO = (96, 176, 84)     # verde
LEGNO = (120, 78, 44)     # marrone

def genera_scene():
    voci = []

    # ---- La casa -------------------------------------------------------------
    img, pennello = nuova_tela(CIELO)                       # sfondo: cielo
    pennello.rectangle([0, 270, LATO, LATO], fill=PRATO)    # prato in basso
    pennello.rectangle([110, 170, 290, 320], fill=(238, 220, 178))  # corpo della casa
    pennello.polygon([(90, 170), (310, 170), (200, 85)], fill=(200, 48, 44))  # tetto rosso
    pennello.rectangle([180, 248, 222, 320], fill=LEGNO)    # porta
    for x in (130, 238):                                    # due finestre
        pennello.rectangle([x, 195, x + 34, 229], fill=(252, 246, 170),
                           outline=LEGNO, width=3)
    voci.append(salva(img, "casa.png", "scene",
                      "Una casa con il tetto rosso, prato verde e cielo azzurro"))

    # ---- L'albero con il sole ------------------------------------------------
    img, pennello = nuova_tela(CIELO)
    pennello.rectangle([0, 285, LATO, LATO], fill=PRATO)
    pennello.ellipse([300, 36, 372, 108], fill=(250, 208, 40))      # sole
    pennello.rectangle([186, 215, 216, 330], fill=LEGNO)            # tronco
    # La chioma: tre ellissi sovrapposte danno una forma "morbida".
    for riquadro in ([118, 92, 284, 232], [96, 142, 212, 254], [190, 142, 306, 254]):
        pennello.ellipse(riquadro, fill=(42, 142, 62))
    voci.append(salva(img, "albero.png", "scene",
                      "Un albero su un prato, con il sole nel cielo"))

    # ---- La barca a vela -----------------------------------------------------
    img, pennello = nuova_tela(CIELO)
    pennello.rectangle([0, 225, LATO, LATO], fill=(36, 110, 190))   # mare
    pennello.polygon([(110, 272), (290, 272), (258, 322), (142, 322)],
                     fill=LEGNO)                                    # scafo (trapezio)
    pennello.line([(200, 128), (200, 272)], fill=(60, 44, 30), width=7)  # albero
    pennello.polygon([(193, 136), (193, 258), (118, 258)],
                     fill=(250, 250, 250))                          # vela
    voci.append(salva(img, "barca.png", "scene",
                      "Una barca a vela sul mare"))

    # ---- L'automobile ----------------------------------------------------------
    img, pennello = nuova_tela(CIELO)
    pennello.rectangle([0, 300, LATO, LATO], fill=(105, 108, 112))  # strada
    for x in range(10, LATO, 60):                                   # striscia tratteggiata
        pennello.rectangle([x, 346, x + 30, 354], fill=BIANCO)
    # rounded_rectangle: come rectangle, ma con gli angoli arrotondati.
    pennello.rounded_rectangle([70, 228, 330, 302], radius=18, fill=(202, 40, 40))  # carrozzeria
    pennello.rounded_rectangle([128, 176, 272, 234], radius=14, fill=(202, 40, 40)) # abitacolo
    for x in (138, 206):                                            # finestrini
        pennello.rectangle([x, 186, x + 56, 228], fill=(198, 228, 246))
    for cx in (132, 268):                                           # ruote
        pennello.ellipse([cx - 32, 270, cx + 32, 334], fill=(28, 28, 28))
        pennello.ellipse([cx - 13, 289, cx + 13, 315], fill=(150, 150, 150))
    voci.append(salva(img, "automobile.png", "scene",
                      "Un'automobile rossa sulla strada"))

    return voci

voci_scene = genera_scene()
print(f"Generate {len(voci_scene)} scene:",
      ", ".join(Path(v["file"]).name for v in voci_scene))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Categoria 3: i GRAFICI (con Matplotlib) — dati di sapore bancario/assicurativo
# ─────────────────────────────────────────────────────────────────────────────
# Qualche convenzione di buona visualizzazione che manteniamo in tutti e quattro:
#   • un solo asse y (mai doppie scale);
#   • griglia leggera e "in secondo piano" (colore chiaro, dietro ai dati);
#   • il colore identifica la serie, il TESTO resta in un grigio quasi nero;
#   • figsize=(4, 4) con dpi=100 → immagine di 400×400 pixel, come le altre
#     (dpi = "dots per inch", punti per pollice: pixel = pollici × dpi).

def _salva_grafico(fig, nome_file, descrizione):
    # Funzione di servizio: il prefisso "_" è la convenzione Python per dire
    # "uso interno". Salviamo la figura e la CHIUDIAMO con plt.close(fig),
    # altrimenti Jupyter mostrerebbe qui sotto tutte le figure aperte.
    percorso = IMG_DIR / nome_file
    fig.savefig(percorso, dpi=100)
    plt.close(fig)
    return {"file": str(percorso), "categoria": "grafici", "descrizione": descrizione}

def genera_grafici():
    voci = []
    plt.rcParams.update({                      # impostazioni comuni alle figure
        "text.color": INCHIOSTRO, "axes.labelcolor": INCHIOSTRO,
        "xtick.color": INCHIOSTRO, "ytick.color": INCHIOSTRO,
        "axes.edgecolor": "#B9B9B9", "font.size": 9,
    })

    # ---- Grafico a barre: nuovi conti correnti per trimestre -------------------
    fig, asse = plt.subplots(figsize=(4, 4), layout="constrained")
    trimestri = ["T1", "T2", "T3", "T4"]
    conti = [1240, 1510, 1385, 1720]
    asse.bar(trimestri, conti, color=PALETTE["teal"])
    asse.set_title("Nuovi conti correnti per trimestre")
    asse.set_ylabel("numero di conti")
    asse.grid(axis="y", color="#DDDDDD", linewidth=0.8)   # griglia solo orizzontale
    asse.set_axisbelow(True)                              # griglia DIETRO le barre
    voci.append(_salva_grafico(fig, "grafico_barre.png",
        "Un grafico a barre: nuovi conti correnti aperti per trimestre"))

    # ---- Grafico a linee: premi assicurativi incassati per mese ----------------
    fig, asse = plt.subplots(figsize=(4, 4), layout="constrained")
    mesi = ["Gen", "Feb", "Mar", "Apr", "Mag", "Giu",
            "Lug", "Ago", "Set", "Ott", "Nov", "Dic"]
    premi = [102, 98, 110, 115, 112, 121, 119, 127, 131, 128, 136, 142]
    asse.plot(mesi, premi, color=PALETTE["purple"], linewidth=2,
              marker="o", markersize=6)
    asse.set_title("Premi incassati per mese")
    asse.set_ylabel("migliaia di €")
    asse.tick_params(axis="x", rotation=60)
    asse.grid(color="#DDDDDD", linewidth=0.8)
    asse.set_axisbelow(True)
    voci.append(_salva_grafico(fig, "grafico_linee.png",
        "Un grafico a linee: andamento mensile dei premi assicurativi incassati"))

    # ---- Grafico a torta: quote di mercato per canale di vendita ---------------
    fig, asse = plt.subplots(figsize=(4, 4), layout="constrained")
    canali = ["Filiale", "Online", "Promotori", "Partner"]
    quote = [38, 27, 22, 13]
    asse.pie(quote, labels=canali, autopct="%1.0f%%",        # etichetta su OGNI fetta
             colors=[PALETTE["teal"], PALETTE["coral"],      # palette in ordine fisso
                     PALETTE["purple"], PALETTE["gray"]],
             startangle=90,
             wedgeprops={"edgecolor": "white", "linewidth": 2})  # fette separate da
    asse.set_title("Quote di mercato per canale")                # un bordo bianco
    voci.append(_salva_grafico(fig, "grafico_torta.png",
        "Un grafico a torta: quote di mercato per canale di vendita"))

    # ---- Grafico a dispersione: età del cliente e importo del prestito ---------
    # "Dispersione" (scatter plot): un punto per osservazione, utile per vedere
    # la RELAZIONE tra due variabili. Generiamo 40 clienti fittizi con il
    # generatore "rng" (seme fisso ⇒ stessi punti a ogni esecuzione).
    fig, asse = plt.subplots(figsize=(4, 4), layout="constrained")
    eta = rng.uniform(25, 70, size=40)                       # 40 età tra 25 e 70 anni
    importo = eta * 800 + rng.normal(0, 6500, size=40) + 8000  # correlata all'età + rumore
    asse.scatter(eta, importo, color=PALETTE["coral"], s=42,
                 edgecolors="white", linewidths=0.8)         # bordo bianco: i punti
    asse.set_title("Età del cliente e importo del prestito") # sovrapposti restano leggibili
    asse.set_xlabel("età (anni)")
    asse.set_ylabel("importo del prestito (€)")
    asse.grid(color="#DDDDDD", linewidth=0.8)
    asse.set_axisbelow(True)
    voci.append(_salva_grafico(fig, "grafico_dispersione.png",
        "Un grafico a dispersione: età del cliente e importo del prestito richiesto"))

    return voci

voci_grafici = genera_grafici()
print(f"Generati {len(voci_grafici)} grafici:",
      ", ".join(Path(v["file"]).name for v in voci_grafici))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Il catalogo completo
# ─────────────────────────────────────────────────────────────────────────────
# Uniamo le tre liste di voci in un unico catalogo. Avere ogni cella che
# RESTITUISCE le proprie voci (invece di accumularle in una lista globale)
# rende il notebook rieseguibile: lanciare due volte una cella non duplica nulla.
catalogo = voci_forme + voci_scene + voci_grafici

# Un DataFrame pandas è la forma più comoda per guardare il catalogo:
# una riga per immagine, una colonna per informazione.
df_catalogo = pd.DataFrame(catalogo)
print(f"Catalogo: {len(df_catalogo)} immagini "
      f"in {df_catalogo['categoria'].nunique()} categorie")
df_catalogo

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Diamo un'occhiata al dataset: la griglia delle immagini
# ─────────────────────────────────────────────────────────────────────────────
# Scriviamo una funzione di visualizzazione RIUTILIZZABILE: ci servirà anche
# più avanti per mostrare i risultati delle ricerche.

def mostra_immagini(percorsi, titoli=None, n_col=4, lato=2.4):
    # percorsi : lista di file immagine da mostrare
    # titoli   : (facoltativo) un titolo per ciascuna immagine
    # n_col    : immagini per riga
    # lato     : dimensione (in pollici) di ogni riquadro
    n = len(percorsi)
    n_rig = math.ceil(n / n_col)                   # righe necessarie (arrotonda in su)
    fig, assi = plt.subplots(n_rig, n_col, figsize=(n_col * lato, n_rig * lato))
    assi = np.atleast_1d(assi).ravel()             # da griglia 2D a lista "piatta"
    for i, asse in enumerate(assi):
        asse.axis("off")                           # niente assi: sono immagini
        if i < n:
            asse.imshow(Image.open(percorsi[i]))   # imshow: mostra un'immagine
            if titoli:
                asse.set_title(titoli[i], fontsize=8, color=INCHIOSTRO)
    plt.tight_layout()
    plt.show()

mostra_immagini([v["file"] for v in catalogo],
                titoli=[Path(v["file"]).name for v in catalogo])

## 2. Dalle immagini ai numeri: gli embedding multimodali di CLIP

Un computer non può confrontare direttamente una frase con un'immagine. Il trucco è trasformare **entrambe** in numeri confrontabili.

- Un **vettore** è una lista ordinata di numeri, ad esempio `[0.12, -0.80, 0.33, ...]`.
- Un **embedding** è un vettore che fa da *riassunto numerico del contenuto*: un modello addestrato apposta (il **modello di embedding**) lo calcola in modo che **contenuti simili producano vettori vicini** tra loro. Nel nostro percorso RAG sul testo abbiamo già usato questa idea; qui facciamo il passo in più.

### CLIP: testo e immagini nello stesso spazio

**CLIP** (*Contrastive Language-Image Pre-training*, OpenAI 2021) è stato addestrato su centinaia di milioni di coppie *(immagine, didascalia)* raccolte dal web, con un addestramento **contrastivo**: il modello impara ad **avvicinare** nello spazio dei vettori ogni immagine alla *sua* didascalia e ad **allontanarla** dalle didascalie sbagliate. Il risultato è uno **spazio vettoriale condiviso**: il vettore della foto di una casa e il vettore della frase *"a house"* finiscono vicini. È questo che rende possibile cercare immagini scrivendo frasi.

### Il problema della lingua (e la soluzione)

C'è un dettaglio pratico importante: l'encoder testuale del CLIP originale capisce bene **quasi solo l'inglese**. Per interrogare in italiano (o in francese) usiamo la sua versione **multilingue**, ottenuta per **distillazione**: un modello "studente" multilingue è stato addestrato a *imitare* gli embedding del "maestro" inglese, frase per frase, in oltre 50 lingue. L'encoder delle **immagini** invece resta quello originale, immutato. Useremo quindi **due** modelli, allineati sullo stesso spazio:

| Ruolo | Modello (da Hugging Face) | Che cosa codifica |
|---|---|---|
| Encoder di immagini | `clip-ViT-B-32` | le immagini dell'archivio (e le immagini-query del §5) |
| Encoder di testo multilingue | `sentence-transformers/clip-ViT-B-32-multilingual-v1` | le nostre domande in italiano |

Entrambi producono vettori di **512 numeri**. Li carichiamo con la libreria `sentence-transformers`, la stessa che usiamo per gli embedding testuali.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Carichiamo i due modelli di embedding
# ─────────────────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import torch   # PyTorch: la libreria di deep learning su cui i modelli girano

# Scegliamo il "dispositivo" di calcolo: "cuda" indica una GPU NVIDIA.
# Se non c'è, va benissimo la CPU: i modelli sono piccoli.
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} — dispositivo di calcolo: {DISPOSITIVO}")

# Alla PRIMA esecuzione questa cella scarica i pesi dei modelli da Hugging Face
# (circa 1 GB in totale) e li mette in cache (~/.cache): dalle volte successive
# il caricamento è immediato e non serve la rete.
modello_immagini = SentenceTransformer("clip-ViT-B-32", device=DISPOSITIVO)
modello_testo = SentenceTransformer(
    "sentence-transformers/clip-ViT-B-32-multilingual-v1", device=DISPOSITIVO)

print("Modelli caricati.")

**📌 Nota sui due warning (entrambi solo informativi)**

**1. Il warning sul token di Hugging Face.** Stiamo scaricando i modelli in forma **anonima** (senza autenticazione), il che per i modelli pubblici è **perfettamente lecito**. Il messaggio ci ricorda solo che, creando un account gratuito e impostando un **token di accesso** (una stringa personale, l'analogo di una chiave API) nella variabile d'ambiente `HF_TOKEN`, otterremmo **rate limit** più alti — cioè un tetto più generoso al numero di richieste accettate dallo stesso indirizzo in un certo intervallo — e download a volte più veloci. Per un singolo scaricamento da ~1 GB è irrilevante; e dal secondo avvio i modelli arrivano dalla cache locale, quindi il tema sparisce del tutto.

C'è un solo scenario in cui il warning diventa concreto: **l'aula contemporanea**. Se quindici partecipanti scaricano i modelli nello stesso momento dalla stessa rete (quindi dallo stesso indirizzo IP pubblico), il rate limit anonimo può scattare e qualche download fallisce. Due rimedi: far impostare un token gratuito a ciascuno (`setx HF_TOKEN "hf_..."` e riavvio di VS Code, come per la chiave OpenAI), oppure — più pratico — far eseguire la cella di caricamento dei modelli a casa, prima della lezione, così ognuno arriva con la cache già piena.

A margine: la riga `PyTorch 2.11.0+cpu — dispositivo di calcolo: cpu` non è un warning ma la nostra stampa diagnostica; conferma che è installata la build solo-CPU, che per questo notebook basta. Sulla workstation con la 5090, volendo la GPU, va installato prima PyTorch con la build CUDA (la nota nel *§Preparazione*) — ma qui non cambierebbe quasi nulla.

**2. La nota «UNEXPECTED».** Quando `transformers` carica un **checkpoint** (il file dei pesi salvati), confronta i nomi dei tensori presenti nel file con quelli che la classe del modello si aspetta: quelli trovati nel file ma non previsti dal modello finiscono nella lista *UNEXPECTED* e vengono semplicemente **ignorati**. Succede tipicamente perché i checkpoint CLIP sono stati salvati con versioni più vecchie della libreria e contengono qualche tensore di servizio (classico esempio: `position_ids`) che le versioni recenti non registrano più, oppure perché il wrapper di `sentence-transformers` usa solo una parte del modello completo. È esattamente il caso che la nota stessa dichiara innocuo: «can be ignored when loading from different task/architecture». Il caso «not ok» sarebbe un altro: se salvassimo *noi* un modello e, ricaricandolo nella *stessa identica* classe, vedessimo chiavi inattese — lì sì sarebbe segno di un'incoerenza tra ciò che abbiamo salvato e ciò che stiamo istanziando. Non è la nostra situazione.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Calcoliamo l'embedding di TUTTE le immagini del catalogo
# ─────────────────────────────────────────────────────────────────────────────
# Il metodo .encode() di sentence-transformers accetta direttamente una lista
# di immagini Pillow (per il modello di immagini) o una lista di frasi (per
# quello di testo). convert_to_numpy=True ci restituisce un array NumPy.

immagini_pil = [Image.open(v["file"]) for v in catalogo]

emb_immagini = modello_immagini.encode(
    immagini_pil,
    convert_to_numpy=True,
    show_progress_bar=False,
)

# La "shape" (forma) di un array dice quante dimensioni ha e quanto è grande
# ciascuna: qui ci aspettiamo (12, 512) = 12 immagini × 512 numeri ciascuna.
print("shape degli embedding:", emb_immagini.shape)
print("primi 6 numeri dell'embedding della prima immagine:",
      np.round(emb_immagini[0, :6], 3))

### Come misuriamo la vicinanza: la similarità coseno

Per dire quanto due vettori $\vec a$ e $\vec b$ sono "vicini" usiamo la **similarità coseno**: il coseno dell'**angolo** tra i due vettori,

$$\cos(\theta) \;=\; \frac{\vec a \cdot \vec b}{\lVert \vec a \rVert \,\lVert \vec b \rVert}$$

dove $\vec a \cdot \vec b$ è il **prodotto scalare** (moltiplichiamo i numeri posizione per posizione e sommiamo tutto) e $\lVert \vec a \rVert$ è la **norma**, cioè la lunghezza del vettore. Dividere per le lunghezze rende la misura indipendente da *quanto sono lunghi* i vettori: conta solo la **direzione**. Il valore va da $-1$ (direzioni opposte) a $+1$ (stessa direzione).

Un trucco che useremo subito: se prima **normalizziamo** i vettori (li riscaliamo a lunghezza 1), il denominatore vale 1 e la similarità coseno **si riduce al semplice prodotto scalare**.

> 📌 **Aspettative sui numeri.** Con i modelli CLIP le similarità assolute sono *basse*: tra una frase e l'immagine "giusta" vedremo tipicamente valori attorno a 0,2–0,35, non certo 0,99. È normale: quello che conta è la **graduatoria** (quale immagine è *più* simile delle altre), non il valore in sé.

## 3. La ricerca «as-is»: similarità calcolata a mano con NumPy

Seguiamo il nostro consueto percorso in due tempi. Prima la versione **as-is** («così com'è»): tutto scritto per esteso, passo per passo, con NumPy, per capire *che cosa succede davvero*. Poi, al §4, la versione **idiomatica**: quella che in produzione scriveremmo davvero, appoggiandoci allo strumento dedicato (un database vettoriale).

Il piano della versione as-is, in quattro mosse:

1. normalizziamo gli embedding delle immagini (lunghezza 1);
2. calcoliamo l'embedding della domanda in italiano, e normalizziamo anche quello;
3. prodotto scalare tra la domanda e *ciascuna* immagine → 12 punteggi di similarità;
4. ordiniamo i punteggi e teniamo i primi *k* (il cosiddetto **top-k**: i *k* risultati migliori).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# La ricerca as-is, passo per passo
# ─────────────────────────────────────────────────────────────────────────────

def normalizza(matrice):
    # np.linalg.norm calcola la lunghezza dei vettori; con axis=-1 la calcola
    # lungo l'ULTIMA dimensione (cioè una lunghezza per ogni riga/vettore) e
    # keepdims=True conserva la dimensione, così la divisione "matrice / norme"
    # si applica riga per riga (è il "broadcasting" di NumPy: l'estensione
    # automatica delle operazioni tra array di forme compatibili).
    return matrice / np.linalg.norm(matrice, axis=-1, keepdims=True)

emb_immagini_norm = normalizza(emb_immagini)

def cerca_numpy(domanda, k=3):
    # 1) embedding della domanda (nota: usiamo il modello di TESTO multilingue)
    emb_domanda = modello_testo.encode([domanda], convert_to_numpy=True)
    emb_domanda = normalizza(emb_domanda)[0]          # un solo vettore di 512 numeri

    # 2) similarità con TUTTE le immagini in un colpo solo: l'operatore @ è il
    #    prodotto matrice-vettore → (12 × 512) @ (512,) = 12 punteggi.
    similarita = emb_immagini_norm @ emb_domanda

    # 3) np.argsort restituisce gli INDICI che ordinerebbero l'array in modo
    #    crescente; lo slicing [::-1] ("dall'ultimo al primo") lo rovescia,
    #    così il primo indice è quello dell'immagine PIÙ simile.
    ordine = np.argsort(similarita)[::-1]

    # 4) teniamo le prime k voci di catalogo, ciascuna col suo punteggio
    return [(catalogo[i], float(similarita[i])) for i in ordine[:k]]

print("Funzione di ricerca pronta.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Prima prova: una domanda in italiano
# ─────────────────────────────────────────────────────────────────────────────
domanda = "una casa con il tetto rosso"
risultati = cerca_numpy(domanda, k=3)

print(f"Domanda: «{domanda}»\n")
for posizione, (voce, punteggio) in enumerate(risultati, start=1):
    print(f"  {posizione}° — similarità {punteggio:.3f} — {Path(voce['file']).name}"
          f"  ({voce['descrizione']})")

# E guardiamoli: riusiamo la funzione di visualizzazione del §1.
mostra_immagini([voce["file"] for voce, _ in risultati],
                titoli=[f"{p}° · sim {s:.3f}"
                        for p, (_, s) in enumerate(risultati, start=1)],
                n_col=3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Panoramica: il primo classificato per quattro domande diverse
# ─────────────────────────────────────────────────────────────────────────────
# Per ogni domanda stampiamo solo il top-1 (k=1): un colpo d'occhio su come
# la ricerca "capisce" colori, forme, oggetti e tipi di grafico.
# Suggerimento: proviamo ad aggiungere domande nostre alla lista.

domande_di_prova = [
    "un grafico a torta",
    "una barca a vela sul mare",
    "una forma geometrica di colore giallo",
    "un veicolo per spostarsi in città",      # nessuna parola in comune col catalogo!
]

for domanda in domande_di_prova:
    (voce, punteggio), = cerca_numpy(domanda, k=1)   # la virgola "spacchetta" l'unico risultato
    print(f"  «{domanda}»".ljust(48),
          f"→  {Path(voce['file']).name}".ljust(28),
          f"(similarità {punteggio:.3f})")

> ⚠️ **E se un risultato è sbagliato?** Può capitare, ed è *materiale didattico prezioso*. Stiamo usando un modello piccolo (il ViT-B-32 è la taglia "base") su disegni stilizzati, e la versione multilingue paga un piccolo pedaggio di qualità rispetto all'inglese. Quando un risultato ci sorprende, guardiamo la graduatoria completa (k=12): spesso l'immagine "giusta" è seconda, a un soffio. Nei sistemi reali si lavora proprio qui: modelli più grandi, descrizioni migliori, *re-ranking* (un secondo modello che riordina i primi risultati — lo abbiamo visto nel percorso RAG testuale).

L'ultima domanda della lista merita una nota: *"un veicolo per spostarsi in città"* non condivide **nessuna parola** con le nostre descrizioni, eppure trova l'automobile. È la differenza tra la ricerca **semantica** (per *significato*, via embedding) e la ricerca per parole chiave: il motivo per cui tutta questa architettura esiste.

## 4. La versione idiomatica: un database vettoriale (ChromaDB)

La versione as-is funziona, ma mostra i suoi limiti appena usciamo dall'aula:

- **ricalcolo**: a ogni riavvio dobbiamo ricalcolare e ricaricare tutti gli embedding — niente **persistenza** (nulla sopravvive su disco);
- **scansione completa**: confrontiamo la domanda con *tutte* le immagini, una per una. Con 12 va benissimo; con 10 milioni no;
- **niente metadati**: non possiamo dire "cerca solo tra i grafici" o "solo nel ramo danni, anno 2025";
- tutta l'infrastruttura (aggiornamenti, API, concorrenza) resterebbe a carico nostro.

Lo strumento dedicato è il **database vettoriale**: un database specializzato nel memorizzare vettori e nel rispondere *molto* in fretta alla domanda "quali sono i *k* vettori più vicini a questo?". La velocità viene da un **indice ANN** (*Approximate Nearest Neighbour*, "vicini più prossimi in modo approssimato"): una struttura — ChromaDB usa **HNSW**, un grafo a più livelli — che trova i vicini quasi sempre giusti senza confrontare tutto con tutto. "Approssimato" significa proprio questo: barattiamo una minuscola probabilità di mancare il vicino esatto in cambio di ricerche enormemente più veloci.

Usiamo **ChromaDB**, che conosciamo già dal percorso RAG testuale: open source, gira in locale, zero configurazione. I suoi concetti base:

| Concetto | Che cos'è |
|---|---|
| **collection** | il contenitore dei dati, l'analogo di una *tabella* |
| **id** | l'identificativo univoco di ogni elemento (useremo il nome del file) |
| **embedding** | il vettore dell'elemento (glielo passiamo noi, già calcolato con CLIP) |
| **metadati** | coppie chiave→valore per filtrare (es. `categoria = "grafici"`) |
| **document** | un testo associato all'elemento (ci mettiamo la descrizione) |

> 📌 **Distanza, non similarità.** Chroma restituisce una **distanza** coseno, definita come $1 - \text{similarità}$: quindi **più piccola = più simile**. Basta ricordarselo (o riconvertire, come faremo noi).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Creiamo il database e la collection
# ─────────────────────────────────────────────────────────────────────────────
import chromadb

# PersistentClient salva tutto nella cartella indicata: alla prossima apertura
# del notebook l'indice sarà GIÀ lì (questa è la persistenza). Per ripartire
# davvero da zero basta cancellare la cartella "chroma_immagini".
cliente = chromadb.PersistentClient(path="chroma_immagini")

# get_or_create_collection: crea la collection se non esiste, altrimenti la
# riapre — anche questa cella, quindi, si può rieseguire senza errori.
collezione = cliente.get_or_create_collection(
    name="immagini_demo",
    # Chiediamo esplicitamente la distanza COSENO per l'indice HNSW
    # (la stessa misura del §3; il default sarebbe la distanza euclidea L2).
    # Nota: su versioni di chromadb precedenti alla 1.0 la sintassi era
    #       metadata={"hnsw:space": "cosine"}.
    configuration={"hnsw": {"space": "cosine"}},
)
print(f"Collection «{collezione.name}» pronta.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Indicizziamo le immagini: id + embedding + metadati + documento
# ─────────────────────────────────────────────────────────────────────────────
# Usiamo upsert ("update or insert"): se un id esiste già lo AGGIORNA, se non
# esiste lo inserisce. L'operazione è quindi idempotente: rieseguire la cella
# non duplica nulla (con .add, invece, otterremmo un errore di id duplicati).

collezione.upsert(
    ids=[Path(v["file"]).name for v in catalogo],          # es. "casa.png"
    embeddings=emb_immagini.tolist(),    # Chroma vuole liste Python, non array
                                         # NumPy: .tolist() fa la conversione.
                                         # Non serve normalizzare: con lo
                                         # spazio "cosine" ci pensa l'indice.
    metadatas=[{"categoria": v["categoria"]} for v in catalogo],
    documents=[v["descrizione"] for v in catalogo],
)

print(f"Elementi nella collection: {collezione.count()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# La ricerca idiomatica: query sulla collection
# ─────────────────────────────────────────────────────────────────────────────

def cerca_chroma(domanda, k=3, filtro=None):
    # 1) embedding della domanda, come al §3 (sempre il modello di testo!)
    emb_domanda = modello_testo.encode([domanda], convert_to_numpy=True)

    # 2) la query al database: n_results è il nostro k; "where" è il filtro
    #    sui metadati (None = nessun filtro). Il resto lo fa l'indice HNSW.
    risposta = collezione.query(
        query_embeddings=emb_domanda.tolist(),
        n_results=k,
        where=filtro,
    )

    # 3) Chroma restituisce liste di liste (una lista per ogni query; noi ne
    #    facciamo una sola, quindi prendiamo l'elemento [0] di ciascun campo)
    #    e le riorganizziamo in una tabella comoda da leggere.
    righe = []
    for id_, dist, meta, doc in zip(risposta["ids"][0], risposta["distances"][0],
                                    risposta["metadatas"][0], risposta["documents"][0]):
        righe.append({
            "file": id_,
            "similarità": round(1 - dist, 3),   # riconvertiamo: sim = 1 − distanza
            "distanza": round(dist, 3),
            "categoria": meta["categoria"],
            "descrizione": doc,
        })
    return righe

# Stessa domanda del §3: ci aspettiamo la STESSA graduatoria (è la nostra
# verifica che la versione idiomatica fa ciò che abbiamo costruito a mano;
# sui decimali l'indice approssimato può introdurre differenze trascurabili).
risultati = cerca_chroma("una casa con il tetto rosso")
mostra_immagini([str(IMG_DIR / r["file"]) for r in risultati],
                titoli=[f"sim {r['similarità']:.3f}" for r in risultati], n_col=3)
pd.DataFrame(risultati)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Il valore aggiunto dei metadati: la ricerca FILTRATA
# ─────────────────────────────────────────────────────────────────────────────
# "Cerca un andamento nel tempo, MA SOLO tra i grafici": il filtro where
# restringe la ricerca agli elementi con categoria = "grafici" PRIMA di
# confrontare gli embedding. In azienda i filtri sono spesso metà del valore:
# per ramo, per anno, per filiale, per livello di riservatezza...

risultati = cerca_chroma("come cambiano i premi incassati nel tempo",
                         k=2,
                         filtro={"categoria": "grafici"})
pd.DataFrame(risultati)

## 5. Ricerca per immagine (*image-to-image*)

Finora la query era una frase. Ma lo spazio è **condiviso**: nulla ci vieta di usare come query... un'**altra immagine**. Il suo embedding (calcolato con l'encoder di *immagini*) vive nello stesso spazio di quelli indicizzati, quindi la ricerca dei più vicini funziona identica.

Casi d'uso tipici nel nostro settore: *"trova le foto simili a questa"* in un archivio di perizie (duplicati tra pratiche diverse: un classico segnale nelle analisi **antifrode**), o il riconoscimento di schemi ricorrenti nei documenti.

Per la prova generiamo un'immagine **nuova**, che *non* esiste nell'archivio — un cerchio **arancione** — e chiediamo al database i suoi **nearest neighbours** («vicini più prossimi»): gli elementi indicizzati col vettore più vicino al suo.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Creiamo l'immagine di query (NON verrà indicizzata)
# ─────────────────────────────────────────────────────────────────────────────
img_query, pennello = nuova_tela()
pennello.ellipse([80, 80, 320, 320], fill=(240, 140, 30))   # arancione
percorso_query = IMG_DIR / "query_cerchio_arancione.png"
img_query.save(percorso_query)

mostra_immagini([percorso_query], titoli=["la nostra query"], n_col=1, lato=2.2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ricerca per immagine: embedding con l'encoder di IMMAGINI, poi stessa query
# ─────────────────────────────────────────────────────────────────────────────
emb_query = modello_immagini.encode([img_query], convert_to_numpy=True)

risposta = collezione.query(query_embeddings=emb_query.tolist(), n_results=3)

file_trovati = risposta["ids"][0]
similarita = [1 - d for d in risposta["distances"][0]]

# Mostriamo la query a sinistra e, a seguire, i tre vicini più prossimi.
# Il cerchio arancione non esiste nell'archivio: ci aspettiamo il cerchio
# rosso al primo posto (stessa forma, colore "caldo" vicino), poi le altre forme.
mostra_immagini([percorso_query] + [str(IMG_DIR / f) for f in file_trovati],
                titoli=["QUERY"] + [f"sim {s:.3f}" for s in similarita],
                n_col=4)

## 6. La «G» di RAG: generiamo la risposta

Chiudiamo il ciclo. Ricapitoliamo dove siamo, nei termini dei tre passi del §0:

1. **Retrieval** ✅ — sappiamo recuperare le immagini pertinenti a una domanda (§3–§5);
2. **Augmentation** — ora *arricchiamo* la domanda: costruiamo un **prompt** (il messaggio di istruzioni e contenuti che passiamo a un modello) allegandoci le immagini recuperate;
3. **Generation** — un modello **multimodale** legge domanda *e* immagini, e risponde.

Il modello che ci serve è un **VLM** (*Vision-Language Model*, modello visione-linguaggio): un modello linguistico che accetta anche immagini in input. Nel prompt gli chiederemo di rispondere **basandosi solo sulle immagini allegate**: è il cosiddetto **grounding** («ancoraggio») della risposta alle fonti, il meccanismo con cui la RAG riduce le **allucinazioni** — le risposte inventate ma dette con sicurezza, il difetto tipico dei modelli lasciati alla sola memoria interna.

Come nel nostro percorso RAG testuale, per la generazione usiamo **gpt-4o-mini** via API OpenAI (accetta immagini). Un'**API** (*Application Programming Interface*) è l'interfaccia con cui un programma usa un servizio remoto; per autenticarci serve una **chiave API**, una stringa segreta personale.

---
> 🔑 **Dove mettiamo la chiave.**<br> 
> Mai nel codice (i notebook si condividono!). La scriviamo in un file `.env`, un file di testo accanto al notebook con la sola riga `OPENAI_API_KEY=sk-....`. La libreria `python-dotenv` la caricherà nell’ambiente al momento dell’esecuzione: niente riavvii. Il `.env` non si condivide e non si mette sotto git (riga `.env` nel `.gitignore`). La cella è comunque protetta: se la chiave non c’è, lo segnala e non esegue la chiamata.
>
> 🔑 La libreria `python.env` legge un file `.env` (testo semplice con righe `NOME=valore`) e ne carica le variabili nell'ambiente del processo Python, tramite la funzione `load_dotenv()`. Così teniamo segreti e configurazioni — come `OPENAI_API_KEY` — in un file accanto al codice invece che nel codice o nell'ambiente di Windows, e `os.environ` le trova come se fossero variabili d'ambiente vere.

---



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Augmentation: costruiamo il prompt multimodale (questa parte è tutta offline)
# ─────────────────────────────────────────────────────────────────────────────
import base64   # base64: codifica che trasforma dati binari (i byte di un PNG)
                # in testo ASCII, il formato con cui le API accettano le immagini

def immagine_in_base64(percorso):
    # Leggiamo i byte del file ("rb" = read binary) e li codifichiamo in base64;
    # .decode() trasforma il risultato da bytes a stringa.
    with open(percorso, "rb") as file_immagine:
        return base64.b64encode(file_immagine.read()).decode()

def prepara_rag(domanda, k=2):
    # 1) RETRIEVAL: le k immagini più pertinenti, dall'indice del §4
    recuperate = cerca_chroma(domanda, k=k)

    # 2) AUGMENTATION: il contenuto del messaggio utente nel formato dell'API
    #    OpenAI — una lista di "blocchi": prima il testo della domanda, poi un
    #    blocco immagine per ciascun file recuperato (come "data URL" base64).
    contenuto = [{
        "type": "text",
        "text": (f"Domanda: {domanda}\n\n"
                 "Rispondi basandoti ESCLUSIVAMENTE sulle immagini allegate, "
                 "recuperate dal nostro archivio."),
    }]
    for r in recuperate:
        dati = immagine_in_base64(IMG_DIR / r["file"])
        contenuto.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{dati}"},
        })
    return recuperate, contenuto

domanda_rag = ("Che cosa mostra il grafico delle quote di mercato per canale? "
               "Qual è il canale principale, e con quale percentuale?")

recuperate, contenuto = prepara_rag(domanda_rag, k=2)

print("Immagini recuperate e allegate al prompt:")
for r in recuperate:
    print(f"  • {r['file']}  (similarità {r['similarità']:.3f}) — {r['descrizione']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Generation: la chiamata al modello multimodale (chiave letta dal file .env)
# ─────────────────────────────────────────────────────────────────────────────
# Un file .env è un semplice file di testo, salvato ACCANTO al notebook, con
# una variabile per riga nel formato NOME=valore (senza spazi attorno all'=):
#
#     OPENAI_API_KEY=sk-...
#
# Rispetto a setx: niente riavvio di VS Code, la chiave vive in un file
# separato dal codice (il notebook si può condividere, il .env no) e per
# cambiare chiave basta modificare il file. Due accortezze:
#   • il nome è esattamente ".env" (solo estensione, senza nome): creiamolo
#     da VS Code, non dal Blocco note, che aggiungerebbe un ".txt" nascosto;
#   • mai sotto controllo di versione: se usiamo git, aggiungiamo la riga
#     ".env" al file .gitignore.
import os

try:
    from dotenv import load_dotenv
    # override=True: carica il .env e, se una variabile esiste GIÀ  nell'ambiente (es. un vecchio setx dimenticato), la sovrascrive
    # con il valore del file: il .env è l'unica fonte di verità.
    if load_dotenv(override=True):          # ← importante, vedi nota sottostante
        print("File .env trovato: variabili caricate.")
    else:
        print("Nessun file .env trovato: proseguiamo con l'ambiente così com'è.")
except ImportError:
    print("⚠️  Libreria python-dotenv non installata (%pip install python-dotenv):")
    print("    proseguiamo con la sola variabile d'ambiente, se presente.")

if not os.environ.get("OPENAI_API_KEY"):
    # Cella "protetta": senza chiave non falliamo, spieghiamo soltanto.
    print("⚠️  OPENAI_API_KEY non trovata (né nel .env né nell'ambiente): saltiamo la chiamata.")
    print("    Creiamo accanto al notebook un file .env con la riga  OPENAI_API_KEY=sk-...")
    print("    e rieseguiamo questa cella (non serve riavviare nulla).")
else:
    from openai import OpenAI

    cliente_llm = OpenAI()   # legge da solo la chiave dalla variabile
                             # d'ambiente, dove load_dotenv l'ha appena messa

    risposta = cliente_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            # Il messaggio "system" fissa le regole del gioco: italiano, e
            # SOLO ciò che si vede nelle immagini (è il grounding del §6).
            {"role": "system",
             "content": ("Sei un assistente per l'analisi di immagini d'archivio. "
                         "Rispondi in italiano basandoti esclusivamente sulle "
                         "immagini allegate; se un'informazione non è visibile "
                         "nelle immagini, dichiaralo esplicitamente.")},
            # Il messaggio "user" è il contenuto multimodale preparato sopra:
            # testo della domanda + immagini recuperate.
            {"role": "user", "content": contenuto},
        ],
        max_tokens=400,      # limite di lunghezza della risposta
    )

    print("Domanda :", domanda_rag)
    print("\nRisposta del modello:\n")
    print(risposta.choices[0].message.content)

---
> 🕵️ **Caso di studio: la chiave-fantasma.** 
>
> **Il sintomo**. Se la cella qui sopra è fallita con `AuthenticationError 401` alla istruzione `risposta = cliente_llm.chat.completions.create(` - il codice 401 significa "autenticazione rifiutata" - nonostante un `.env` corretto (una chiave nuova, completa e corretta), cosa significa?
>
> **L'indagine.** Una chiave non si stampa mai per intero; ne stampiamo l'*impronta* (lunghezza, inizio, ultimi 4 caratteri):
>
> ```python
>   import os
>   chiave = os.environ.get("OPENAI_API_KEY", "")
>   print(len(chiave), chiave[:12] + "…", chiave[-4:])
> ```
>
> <u>Confronto</u> degli gli ultimi 4 caratteri con quelli della chiave scritta nel `.env`. <u>Possibile verdetto</u>: **non coincidono!!** ⟶ verso OpenAI sta partendo un'*altra* chiave.
>
> **Il colpevole.** Una chiave vecchia e ormai revocata, impostata tempo prima con `setx` (il comando Windows che salva una variabile d'ambiente in modo permanente nel profilo utente), era rimasta lì: una **chiave-fantasma** — così chiamiamo una variabile che non dovrebbe più esistere ma continua ad abitare l'ambiente. Ogni programma, all'avvio, *eredita* una copia delle variabili d'ambiente: VS Code l'ha ereditata e, poiché `load_dotenv()` per default **non sovrascrive** una variabile già presente (`override=False`), il `.env` — giusto — veniva ignorato in silenzio.
>
> **Il rimedio.** Due mosse:<br>
> (1) `load_dotenv(override=True)`, così il file vince sempre sull'ambiente;<br>
> (2) pulizia definitiva da PowerShell con `[Environment]::SetEnvironmentVariable("OPENAI_API_KEY", $null, "User")` — da copiare alla lettera, `Environment` è il nome reale della classe .NET  (non è un segnaposto!) — e riavvio di VS Code.
>
> **La morale.** Davanti a un'autenticazione che fallisce "senza motivo", la prima domanda non è *«la chiave è giusta?»* ma *«**quale** chiave sta davvero partendo, e da dove viene?»*. Le fonti possibili sono più d'una (ambiente di sistema, ambiente del processo, `.env`), e la precedenza tra loro va resa esplicita: con `override=True` dichiariamo che **il `.env` è l'unica fonte di verità**.

---

In [ ]:
import os
chiave = os.environ.get("OPENAI_API_KEY", "")
print("lunghezza          :", len(chiave))
print("inizio             :", chiave[:12] + "…")
print("ultimi 4 caratteri :", chiave[-4:])
print("caratteri sospetti :", any(c in chiave for c in ' "\''))   # spazi o virgolette

## 7. Per andare oltre: esercizi e varianti

1. **Le nostre immagini.** Copiamo qualche foto nella cartella `immagini/`, aggiungiamo a mano le voci al `catalogo` (file, categoria, descrizione), ricalcoliamo gli embedding (§2) e rifacciamo l'`upsert` (§4). Le foto reali, tra l'altro, funzionano in genere *meglio* dei nostri disegni: CLIP è stato addestrato soprattutto su fotografie.

2. **Query in francese.** Il modello di testo è multilingue: proviamo `cerca_chroma("une maison au toit rouge")` oppure `"un graphique en secteurs"` (il grafico a torta). Stesso indice, nessuna modifica.

3. **La soglia di pertinenza.** Oggi restituiamo *sempre* k risultati, anche per una domanda assurda ("un elefante rosa"): proviamo e osserviamo i punteggi. Modifichiamo poi `cerca_chroma` perché scarti i risultati sotto una similarità minima e risponda "nessuna immagine pertinente". Come scegliamo la soglia? Sperimentando su domande "giuste" e "sbagliate": non esiste un valore universale.

4. **Un encoder più grande.** Esiste `clip-ViT-L-14`, più accurato. ⚠️ Attenzione però: il nostro modello di testo multilingue è **allineato solo allo spazio del B-32**. Se cambiamo l'encoder di immagini dobbiamo cambiare coerentemente quello di testo (o passare a query in inglese con l'encoder nativo del modello scelto). Due modelli non allineati producono vettori *non confrontabili*: le similarità diventano rumore.

5. **L'architettura alternativa: RAG "per didascalie".** Invece di indicizzare gli embedding delle immagini, potremmo far *descrivere* ogni immagine a un VLM e indicizzare **le didascalie** con la pipeline testuale che già conosciamo. Pro: riusiamo tutto lo stack del testo, con modelli di embedding testuali eccellenti. Contro: ciò che la didascalia non dice è perso per sempre (la ricerca vede la descrizione, non l'immagine). Molti sistemi di produzione combinano i due approcci.

6. **Metadati più ricchi.** Aggiungiamo ai metadati `anno`, `ramo`, `filiale` e proviamo filtri combinati nella `where` — è la funzionalità che in azienda trasforma la demo in uno strumento.

## In sintesi

- Con **CLIP** portiamo immagini e frasi nello **stesso spazio vettoriale**: la ricerca di immagini diventa una ricerca di vettori vicini. Per le domande in italiano usiamo l'encoder di testo **multilingue** allineato.
- La versione **as-is** (NumPy) ci ha mostrato i meccanismi: normalizzazione, prodotto scalare, ordinamento. La versione **idiomatica** (ChromaDB) aggiunge ciò che serve davvero in produzione: **persistenza**, **velocità** (indice HNSW) e **filtri sui metadati**.
- La query può essere una frase *o* un'immagine: stesso spazio, stessa ricerca.
- La «G» chiude il ciclo: **retrieval → augmentation → generation**, con un VLM che risponde **ancorato** alle immagini recuperate.

### Piccolo glossario

| Termine | In una riga |
|---|---|
| RAG | Retrieval-Augmented Generation: recupero dei contenuti pertinenti + risposta generata su quei contenuti |
| multimodale | che tratta più tipi di dato (testo, immagini, audio...) |
| embedding | vettore di numeri che riassume un contenuto; contenuti simili → vettori vicini |
| CLIP | modello (addestramento contrastivo su coppie immagine-didascalia) che mette testo e immagini nello stesso spazio |
| distillazione | addestrare un modello "studente" a imitare gli output di un "maestro" |
| similarità coseno | coseno dell'angolo tra due vettori; con vettori normalizzati è il prodotto scalare |
| normalizzazione | riscalare un vettore a lunghezza 1 |
| top-k | i k risultati migliori di una ricerca |
| database vettoriale | database specializzato nella ricerca dei vettori più vicini |
| collection | la "tabella" di un database vettoriale |
| upsert | update-or-insert: inserisce o aggiorna; rieseguirlo non duplica (operazione *idempotente*) |
| HNSW / ANN | indice a grafo per la ricerca *approssimata* dei vicini più prossimi, molto veloce |
| distanza coseno | 1 − similarità coseno: più piccola = più simile |
| metadati | coppie chiave→valore associate a un elemento, usate per filtrare |
| nearest neighbours | i "vicini più prossimi": gli elementi col vettore più vicino alla query |
| VLM | Vision-Language Model: modello linguistico che accetta anche immagini |
| prompt | il messaggio (istruzioni + contenuti) che passiamo al modello |
| grounding | ancorare la risposta alle sole fonti fornite |
| allucinazione | risposta inventata ma plausibile di un modello |
| ground truth | la "verità di riferimento" con cui giudichiamo i risultati |
| dataset sintetico | dataset generato artificialmente |
| seed (seme) | valore che fissa i numeri casuali, rendendo l'esecuzione riproducibile |
| variabile d'ambiente | valore del sistema operativo reso disponibile ai programmi (es. le chiavi API) |
| base64 | codifica di dati binari in testo, usata per allegare immagini alle API |

---

*© 2026 Antonio Piemontese — materiale didattico.*